In [ ]:
import os
import re
import sys
import json
import time
import hashlib
import platform
import warnings

import numpy as np
import pandas as pd
import joblib
import sklearn
import matplotlib

import tkinter as tk
from tkinter import ttk, messagebox, filedialog

from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold, train_test_split, cross_val_predict
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler


warnings.filterwarnings("ignore")

if sys.platform.startswith("win"):
    try:
        import ctypes
        ctypes.windll.shcore.SetProcessDpiAwareness(1)
    except Exception:
        pass


DATA_PATH = (
    r"D:\2026 Work\My Papers\waste glass concrete"
    r"\Revision of Manuscript\1st Revision\Work\Data"
    r"\01_WGP_database_with_ids.csv"
)

SAVE_DIR = (
    r"D:\2026 Work\My Papers\waste glass concrete"
    r"\Revision of Manuscript\1st Revision\Work\Results\ExtraTrees_GUI"
)

TARGET = "CS"
UNIT = "MPa"
CURING_AGE_COLUMN = "Days"
ID_COLUMNS = ["record_id", "source_id", "source"]
EXTRA_EXCLUDED_COLUMNS = []

TEST_SIZE = 0.20
RANDOM_STATE = 42
CV_FOLDS = 5
ALPHA = 0.05
INTERVAL_HALF_WIDTH_OVERRIDE = None

NN_CAUTION_PERCENTILE = 95
NN_LIMIT_PERCENTILE = 99

ET_PARAMS = {
    "n_estimators": 200,
    "min_samples_split": 10,
    "min_samples_leaf": 2,
    "max_features": 1.0,
    "max_depth": 20,
    "bootstrap": False,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}


def read_csv_any(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError("Unable to decode the CSV file.")


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_age_feature(names):
    if CURING_AGE_COLUMN and CURING_AGE_COLUMN in names:
        return CURING_AGE_COLUMN
    patterns = (
        r"curing",
        r"(^|[^a-z])age([^a-z]|$)",
        r"(^|[^a-z])days?([^a-z]|$)",
    )
    for pattern in patterns:
        for name in names:
            if re.search(pattern, name.lower()):
                return name
    return None


def find_id_columns(names):
    if ID_COLUMNS is not None:
        return [c for c in ID_COLUMNS if c in names]
    pattern = r"(^|[^a-z])(id|ids|identifier|ref|reference|source|study|paper|no)([^a-z]|$)"
    found = []
    for name in names:
        lowered = name.lower().strip()
        if lowered in ("id", "index", "s/n", "sn", "sl", "sl no", "serial") or re.search(pattern, lowered):
            found.append(name)
    return found


def conformal_quantile(scores, alpha):
    scores = np.sort(np.asarray(scores, dtype=float))
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    if k > n:
        return float(scores[-1])
    return float(scores[k - 1])


def fmt(value, digits=4):
    if value is None:
        return "N/A"
    try:
        value = float(value)
    except (TypeError, ValueError):
        return str(value)
    if not np.isfinite(value):
        return "N/A"
    return f"{value:.{digits}g}"


class CompressiveStrengthApp:

    def __init__(self):
        self.colors = {
            "navy": "#17324D",
            "blue": "#2878B5",
            "light_blue": "#EAF3F9",
            "teal": "#1F8A8A",
            "light_teal": "#E8F6F6",
            "green": "#2E8B57",
            "light_green": "#EAF6EF",
            "orange": "#C46A12",
            "light_orange": "#FFF3E8",
            "red": "#C0392B",
            "light_red": "#FCECEA",
            "purple": "#6C5CE7",
            "light_purple": "#F0EEFF",
            "background": "#F4F7FA",
            "card": "#FFFFFF",
            "border": "#D9E1E8",
            "text": "#243447",
            "muted": "#6B7785",
            "white": "#FFFFFF",
        }

        os.makedirs(SAVE_DIR, exist_ok=True)
        self.bundle_path = os.path.join(SAVE_DIR, "ExtraTrees_CS_pipeline.joblib")
        self.record_path = os.path.join(SAVE_DIR, "ExtraTrees_CS_run_record.json")
        self.test_pred_path = os.path.join(SAVE_DIR, "ExtraTrees_CS_test_predictions.csv")
        self.history_path = os.path.join(SAVE_DIR, "CS_prediction_history.json")

        self.history = []
        self.current = None

        self.load_data()
        self.fit_or_load()
        self.evaluate()
        self.write_run_record()
        self.build_gui()
        self.load_history()

    def load_data(self):
        if not os.path.exists(DATA_PATH):
            raise FileNotFoundError(f"Dataset not found:\n{DATA_PATH}")

        self.data_hash = file_sha256(DATA_PATH)
        df = read_csv_any(DATA_PATH)
        df.columns = df.columns.astype(str).str.strip()

        if TARGET not in df.columns:
            raise ValueError(
                f"Target column '{TARGET}' not found.\n\n"
                "Available columns:\n" + "\n".join(df.columns)
            )

        rows_in_file = len(df)
        df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
        df = df.dropna(subset=[TARGET])
        rows_missing_target = rows_in_file - len(df)

        candidates = [c for c in df.columns if c != TARGET]
        self.id_columns = find_id_columns(candidates)
        excluded = self.id_columns + [
            c for c in EXTRA_EXCLUDED_COLUMNS if c in candidates and c not in self.id_columns
        ]
        self.excluded_columns = excluded

        rows_before = len(df)
        modelling_columns = [c for c in df.columns if c not in excluded]
        df = df.loc[~df.duplicated(subset=modelling_columns)].reset_index(drop=True)
        duplicate_rows = rows_before - len(df)

        empty_columns = [
            c for c in modelling_columns if c != TARGET and df[c].isna().all()
        ]
        self.ids = df[self.id_columns].copy() if self.id_columns else pd.DataFrame(index=df.index)

        X = df.drop(columns=[TARGET] + excluded + empty_columns).copy()
        y = df[TARGET].astype(float)

        self.numeric_features = []
        self.categorical_features = []

        for column in X.columns:
            if pd.api.types.is_numeric_dtype(X[column]):
                X[column] = X[column].replace([np.inf, -np.inf], np.nan).astype(float)
                self.numeric_features.append(column)
                continue
            converted = pd.to_numeric(X[column], errors="coerce")
            present = X[column].notna().sum()
            if present > 0 and converted.notna().sum() >= 0.9 * present:
                X[column] = converted.replace([np.inf, -np.inf], np.nan).astype(float)
                self.numeric_features.append(column)
            else:
                X[column] = X[column].map(
                    lambda v: str(v).strip() if pd.notna(v) else np.nan
                )
                self.categorical_features.append(column)

        self.feature_names = list(X.columns)
        self.X_full = X
        self.y_full = y

        (
            self.X_train,
            self.X_test,
            self.y_train,
            self.y_test,
        ) = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

        self.age_feature = find_age_feature(self.feature_names)

        self.feature_stats = {}
        for column in self.numeric_features:
            values = self.X_train[column].dropna()
            self.feature_stats[column] = {
                "type": "numeric",
                "min": float(values.min()),
                "max": float(values.max()),
                "median": float(values.median()),
                "mean": float(values.mean()),
            }
        for column in self.categorical_features:
            values = self.X_train[column].dropna()
            categories = sorted(values.unique().tolist())
            mode = values.mode().iloc[0] if len(values) else ""
            self.feature_stats[column] = {
                "type": "categorical",
                "categories": categories,
                "mode": mode,
            }

        self.data_summary = {
            "rows_in_file": int(rows_in_file),
            "rows_missing_target_removed": int(rows_missing_target),
            "id_columns_excluded": self.id_columns,
            "other_columns_excluded": [c for c in excluded if c not in self.id_columns],
            "duplicate_rows_removed": int(duplicate_rows),
            "empty_columns_removed": empty_columns,
            "rows_used": int(len(df)),
            "training_rows": int(len(self.X_train)),
            "test_rows": int(len(self.X_test)),
        }

    def build_pipeline(self):
        transformers = []
        if self.numeric_features:
            transformers.append(
                ("num", SimpleImputer(strategy="median"), self.numeric_features)
            )
        if self.categorical_features:
            transformers.append(
                (
                    "cat",
                    Pipeline(
                        [
                            ("impute", SimpleImputer(strategy="most_frequent")),
                            (
                                "encode",
                                OrdinalEncoder(
                                    handle_unknown="use_encoded_value",
                                    unknown_value=-1,
                                ),
                            ),
                        ]
                    ),
                    self.categorical_features,
                )
            )
        preprocess = ColumnTransformer(transformers, remainder="drop")
        return Pipeline(
            [
                ("preprocess", preprocess),
                ("model", ExtraTreesRegressor(**ET_PARAMS)),
            ]
        )

    def signature(self):
        return {
            "data_sha256": self.data_hash,
            "excluded_columns": self.excluded_columns,
            "features": self.feature_names,
            "categorical_features": self.categorical_features,
            "et_params": dict(ET_PARAMS),
            "test_size": TEST_SIZE,
            "random_state": RANDOM_STATE,
            "cv_folds": CV_FOLDS,
            "alpha": ALPHA,
            "sklearn": sklearn.__version__,
        }

    def fit_or_load(self):
        signature = self.signature()

        if os.path.exists(self.bundle_path):
            try:
                bundle = joblib.load(self.bundle_path)
                if bundle.get("signature") == signature:
                    self.apply_bundle(bundle)
                    self.model_source = "loaded from saved pipeline"
                    return
            except Exception:
                pass

        start = time.time()
        pipeline = self.build_pipeline().fit(self.X_train, self.y_train)
        fit_seconds = time.time() - start

        folds = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        oof = cross_val_predict(
            self.build_pipeline(), self.X_train, self.y_train, cv=folds
        )
        y_train = self.y_train.to_numpy()
        scores = np.abs(y_train - oof)
        half_width = conformal_quantile(scores, ALPHA)

        z_train = pipeline.named_steps["preprocess"].transform(self.X_train)
        scaler = StandardScaler().fit(z_train)
        z_scaled = scaler.transform(z_train)
        neighbours = NearestNeighbors(n_neighbors=2).fit(z_scaled)
        reference = neighbours.kneighbors(z_scaled)[0][:, 1]

        bundle = {
            "signature": signature,
            "pipeline": pipeline,
            "interval_half_width": half_width,
            "oof_r2": float(r2_score(y_train, oof)),
            "oof_rmse": float(np.sqrt(mean_squared_error(y_train, oof))),
            "nn_scaler": scaler,
            "nn_model": neighbours,
            "nn_caution": float(np.percentile(reference, NN_CAUTION_PERCENTILE)),
            "nn_limit": float(np.percentile(reference, NN_LIMIT_PERCENTILE)),
            "fit_seconds": float(fit_seconds),
            "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        }

        joblib.dump(bundle, self.bundle_path)
        self.apply_bundle(bundle)
        self.model_source = "trained in this session"

    def apply_bundle(self, bundle):
        self.bundle = bundle
        self.pipeline = bundle["pipeline"]
        self.preprocess = self.pipeline.named_steps["preprocess"]
        self.model = self.pipeline.named_steps["model"]
        self.q = float(bundle["interval_half_width"])
        if INTERVAL_HALF_WIDTH_OVERRIDE is not None:
            self.q = float(INTERVAL_HALF_WIDTH_OVERRIDE)
        self.nn_scaler = bundle["nn_scaler"]
        self.nn_model = bundle["nn_model"]
        self.nn_caution = bundle["nn_caution"]
        self.nn_limit = bundle["nn_limit"]

    def nearest_distance(self, X):
        z = self.nn_scaler.transform(self.preprocess.transform(X))
        distance, index = self.nn_model.kneighbors(z, n_neighbors=1)
        return distance[:, 0], index[:, 0]

    def out_of_range(self, row):
        flagged = []
        for column in self.numeric_features:
            value = row[column]
            if pd.isna(value):
                continue
            stats = self.feature_stats[column]
            if value < stats["min"] or value > stats["max"]:
                flagged.append((column, float(value), stats["min"], stats["max"]))
        return flagged

    def evaluate(self):
        y_true = self.y_test.to_numpy()
        y_pred = self.pipeline.predict(self.X_test)
        y_train_pred = self.pipeline.predict(self.X_train)

        self.y_test_pred = y_pred
        self.test_r2 = float(r2_score(y_true, y_pred))
        self.test_rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        self.test_mae = float(mean_absolute_error(y_true, y_pred))
        nonzero = np.abs(y_true) > 1e-12
        self.test_mape = (
            float(np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100)
            if nonzero.any()
            else float("nan")
        )
        self.train_r2 = float(r2_score(self.y_train, y_train_pred))

        lower = y_pred - self.q
        upper = y_pred + self.q
        covered = (y_true >= lower) & (y_true <= upper)
        self.test_coverage = float(np.mean(covered))
        self.interval_width = float(2 * self.q)

        distances, _ = self.nearest_distance(self.X_test)
        range_flags = np.array(
            [len(self.out_of_range(self.X_test.iloc[i])) > 0 for i in range(len(self.X_test))]
        )
        self.test_beyond_limit = int(np.sum(distances > self.nn_limit))
        self.test_out_of_range = int(np.sum(range_flags))

        table = self.X_test.copy()
        table.insert(0, "nn_distance", distances)
        table.insert(0, "covered", covered)
        table.insert(0, "upper_95", upper)
        table.insert(0, "lower_95", lower)
        table.insert(0, "predicted_CS", y_pred)
        table.insert(0, "actual_CS", y_true)
        for column in reversed(self.id_columns):
            table.insert(0, column, self.ids.loc[self.X_test.index, column].to_numpy())
        table.to_csv(self.test_pred_path, index=True, index_label="row_index")

    def write_run_record(self):
        record = {
            "created": time.strftime("%Y-%m-%d %H:%M:%S"),
            "model": "ExtraTreesRegressor in a scikit-learn Pipeline",
            "model_source": self.model_source,
            "data": {
                "path": DATA_PATH,
                "sha256": self.data_hash,
                "target": TARGET,
                "features": self.feature_names,
                "categorical_features": self.categorical_features,
                "curing_age_feature": self.age_feature,
                **self.data_summary,
            },
            "split": {
                "method": "train_test_split",
                "test_size": TEST_SIZE,
                "random_state": RANDOM_STATE,
            },
            "preprocessing": {
                "numeric": "median imputation fitted on training data",
                "categorical": "most-frequent imputation and ordinal encoding fitted on training data",
            },
            "hyperparameters": dict(ET_PARAMS),
            "prediction_interval": {
                "method": f"cross-conformal, {CV_FOLDS}-fold out-of-fold absolute residuals on training data",
                "nominal_coverage": 1 - ALPHA,
                "half_width": self.q,
                "override_applied": INTERVAL_HALF_WIDTH_OVERRIDE is not None,
                "test_empirical_coverage": self.test_coverage,
                "width": self.interval_width,
            },
            "applicability_domain": {
                "univariate": "training minimum and maximum of each numeric input",
                "joint": "Euclidean nearest-neighbour distance in standardised preprocessed input space",
                "caution_percentile": NN_CAUTION_PERCENTILE,
                "caution_distance": self.nn_caution,
                "limit_percentile": NN_LIMIT_PERCENTILE,
                "limit_distance": self.nn_limit,
                "test_rows_beyond_limit": self.test_beyond_limit,
                "test_rows_out_of_range": self.test_out_of_range,
                "feature_ranges": self.feature_stats,
            },
            "performance": {
                "train_R2": self.train_r2,
                "cv_oof_R2": self.bundle["oof_r2"],
                "cv_oof_RMSE": self.bundle["oof_rmse"],
                "test_R2": self.test_r2,
                "test_RMSE": self.test_rmse,
                "test_MAE": self.test_mae,
                "test_MAPE_percent": self.test_mape if np.isfinite(self.test_mape) else None,
            },
            "environment": {
                "python": platform.python_version(),
                "platform": platform.platform(),
                "numpy": np.__version__,
                "pandas": pd.__version__,
                "scikit_learn": sklearn.__version__,
                "matplotlib": matplotlib.__version__,
                "joblib": joblib.__version__,
            },
            "files": {
                "pipeline": self.bundle_path,
                "test_predictions": self.test_pred_path,
            },
        }
        with open(self.record_path, "w", encoding="utf-8") as handle:
            json.dump(record, handle, indent=4, default=str)

    def build_gui(self):
        self.root = tk.Tk()
        self.root.title("Compressive Strength Predictor")
        self.root.geometry("1450x920")
        self.root.minsize(1150, 760)
        self.root.configure(bg=self.colors["background"])
        self.configure_ttk()

        header = tk.Frame(self.root, bg=self.colors["navy"], height=84)
        header.pack(fill="x")
        header.pack_propagate(False)

        title_frame = tk.Frame(header, bg=self.colors["navy"])
        title_frame.pack(side="left", padx=28, pady=12)
        tk.Label(
            title_frame,
            text="Compressive Strength Predictor",
            font=("Segoe UI", 20, "bold"),
            fg=self.colors["white"],
            bg=self.colors["navy"],
        ).pack(anchor="w")
        tk.Label(
            title_frame,
            text="Waste glass powder concrete | Extra Trees with 95% prediction interval",
            font=("Segoe UI", 10),
            fg="#D5E5F2",
            bg=self.colors["navy"],
        ).pack(anchor="w", pady=(3, 0))

        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill="both", expand=True, padx=14, pady=12)

        self.notebook.add(self.create_prediction_tab(), text="  Prediction  ")
        self.notebook.add(self.create_domain_tab(), text="  Validated Domain  ")
        self.notebook.add(self.create_analysis_tab(), text="  Analysis  ")
        self.notebook.add(self.create_history_tab(), text="  History  ")

        status = tk.Frame(self.root, bg=self.colors["navy"], height=30)
        status.pack(fill="x")
        status.pack_propagate(False)
        self.status_var = tk.StringVar(value=f"Model ready ({self.model_source})")
        tk.Label(
            status,
            textvariable=self.status_var,
            font=("Segoe UI", 9),
            fg=self.colors["white"],
            bg=self.colors["navy"],
        ).pack(side="left", padx=15)

        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)

    def configure_ttk(self):
        style = ttk.Style()
        try:
            style.theme_use("clam")
        except Exception:
            pass
        style.configure(".", font=("Segoe UI", 10))
        style.configure("TNotebook", background=self.colors["background"], borderwidth=0)
        style.configure("TNotebook.Tab", padding=(20, 10), font=("Segoe UI", 10, "bold"))
        style.configure("TFrame", background=self.colors["background"])
        style.configure("TButton", font=("Segoe UI", 10, "bold"), padding=(12, 8))
        style.configure(
            "Accent.TButton",
            foreground=self.colors["white"],
            background=self.colors["blue"],
            font=("Segoe UI", 11, "bold"),
            padding=(18, 10),
        )
        style.map("Accent.TButton", background=[("active", self.colors["navy"])])
        style.configure("TEntry", padding=6)
        style.configure("TCombobox", padding=5)
        style.configure("Treeview", rowheight=26, font=("Segoe UI", 9))
        style.configure("Treeview.Heading", font=("Segoe UI", 9, "bold"))

    def card(self, parent):
        return tk.Frame(
            parent,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1,
        )

    def create_prediction_tab(self):
        tab = ttk.Frame(self.notebook)
        main = tk.Frame(tab, bg=self.colors["background"])
        main.pack(fill="both", expand=True, padx=12, pady=12)

        left = self.card(main)
        left.pack(side="left", fill="both", expand=True, padx=(0, 8))

        tk.Label(
            left,
            text="Input Parameters",
            font=("Segoe UI", 15, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["card"],
        ).pack(anchor="w", padx=20, pady=(18, 2))
        tk.Label(
            left,
            text="Grey ranges show the training domain. Values outside it are marked in red.",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["card"],
        ).pack(anchor="w", padx=20, pady=(0, 10))

        canvas = tk.Canvas(left, bg=self.colors["card"], highlightthickness=0)
        scrollbar = ttk.Scrollbar(left, orient="vertical", command=canvas.yview)
        inputs = tk.Frame(canvas, bg=self.colors["card"])
        inputs.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        window = canvas.create_window((0, 0), window=inputs, anchor="nw")
        canvas.bind("<Configure>", lambda e: canvas.itemconfig(window, width=e.width))
        canvas.configure(yscrollcommand=scrollbar.set)
        canvas.pack(side="left", fill="both", expand=True, padx=(15, 0))
        scrollbar.pack(side="right", fill="y", pady=5)

        quick = tk.Frame(
            inputs,
            bg=self.colors["light_blue"],
            highlightbackground="#C9DFEF",
            highlightthickness=1,
        )
        quick.pack(fill="x", padx=5, pady=(0, 12))
        tk.Label(
            quick,
            text="Training samples",
            font=("Segoe UI", 9, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["light_blue"],
        ).pack(side="left", padx=10, pady=8)
        ttk.Button(quick, text="Random", command=self.fill_random_sample).pack(side="left", padx=3, pady=5)
        ttk.Button(quick, text="Minimum CS", command=self.fill_min_case).pack(side="left", padx=3, pady=5)
        ttk.Button(quick, text="Maximum CS", command=self.fill_max_case).pack(side="left", padx=3, pady=5)

        self.entries = {}
        self.range_labels = {}

        for feature in self.feature_names:
            row = tk.Frame(inputs, bg=self.colors["card"])
            row.pack(fill="x", padx=5, pady=4)
            tk.Label(
                row,
                text=feature,
                font=("Segoe UI", 9, "bold"),
                fg=self.colors["text"],
                bg=self.colors["card"],
                anchor="w",
                width=30,
            ).pack(side="left")

            stats = self.feature_stats[feature]
            if stats["type"] == "categorical":
                combo = ttk.Combobox(row, width=22, state="readonly", values=stats["categories"])
                combo.set(stats["mode"])
                combo.pack(side="left", padx=6)
                self.entries[feature] = combo
            else:
                entry = ttk.Entry(row, width=16)
                entry.insert(0, fmt(stats["median"], 6))
                entry.pack(side="left", padx=6)
                entry.bind("<KeyRelease>", self.check_ranges)
                label = tk.Label(
                    row,
                    text=self.range_text(feature),
                    font=("Segoe UI", 8),
                    fg=self.colors["muted"],
                    bg=self.colors["card"],
                )
                label.pack(side="left")
                self.entries[feature] = entry
                self.range_labels[feature] = label

        buttons = tk.Frame(inputs, bg=self.colors["card"])
        buttons.pack(fill="x", padx=5, pady=18)
        ttk.Button(buttons, text="PREDICT CS", style="Accent.TButton", command=self.predict).pack(side="left", padx=3)
        ttk.Button(buttons, text="Reset", command=self.clear_inputs).pack(side="left", padx=5)

        right = self.card(main)
        right.pack(side="right", fill="both", expand=True, padx=(8, 0))

        tk.Label(
            right,
            text="Prediction Result",
            font=("Segoe UI", 15, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["card"],
        ).pack(anchor="w", padx=22, pady=(18, 0))
        tk.Label(
            right,
            text="Estimated compressive strength",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["card"],
        ).pack(anchor="w", padx=22)

        result = tk.Frame(
            right,
            bg=self.colors["light_blue"],
            highlightbackground="#C9DFEF",
            highlightthickness=1,
        )
        result.pack(fill="x", padx=22, pady=(16, 10))

        self.result_var = tk.StringVar(value="READY")
        tk.Label(
            result,
            textvariable=self.result_var,
            font=("Segoe UI", 34, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["light_blue"],
        ).pack(pady=(22, 0))
        tk.Label(
            result,
            text=f"CS ({UNIT})",
            font=("Segoe UI", 11),
            fg=self.colors["muted"],
            bg=self.colors["light_blue"],
        ).pack(pady=(0, 10))

        self.interval_var = tk.StringVar(value="")
        tk.Label(
            result,
            textvariable=self.interval_var,
            font=("Segoe UI", 11, "bold"),
            fg=self.colors["blue"],
            bg=self.colors["light_blue"],
        ).pack(pady=(0, 4))

        self.age_var = tk.StringVar(value="")
        tk.Label(
            result,
            textvariable=self.age_var,
            font=("Segoe UI", 10),
            fg=self.colors["text"],
            bg=self.colors["light_blue"],
        ).pack(pady=(0, 18))

        self.domain_card = tk.Frame(
            right,
            bg=self.colors["light_blue"],
            highlightbackground=self.colors["border"],
            highlightthickness=1,
        )
        self.domain_card.pack(fill="x", padx=22, pady=(0, 10))

        self.domain_title = tk.Label(
            self.domain_card,
            text="Domain check",
            font=("Segoe UI", 11, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["light_blue"],
            anchor="w",
        )
        self.domain_title.pack(fill="x", padx=14, pady=(10, 2))

        self.domain_text = tk.Label(
            self.domain_card,
            text="Run a prediction to check the input against the validated domain.",
            font=("Segoe UI", 9),
            fg=self.colors["text"],
            bg=self.colors["light_blue"],
            justify="left",
            anchor="w",
            wraplength=520,
        )
        self.domain_text.pack(fill="x", padx=14, pady=(0, 12))
        self.domain_card.bind(
            "<Configure>",
            lambda e: self.domain_text.configure(wraplength=max(200, e.width - 40)),
        )

        tk.Label(
            right,
            text=(
                f"Interval: {int((1 - ALPHA) * 100)}% cross-conformal, half-width "
                f"{self.q:.2f} {UNIT}; test coverage {self.test_coverage * 100:.1f}%"
            ),
            font=("Segoe UI", 8),
            fg=self.colors["muted"],
            bg=self.colors["card"],
        ).pack(anchor="w", padx=22)

        actions = tk.Frame(right, bg=self.colors["card"])
        actions.pack(fill="x", padx=22, pady=10)
        ttk.Button(actions, text="Save Result", command=self.save_result).pack(side="left", padx=(0, 5))
        ttk.Button(actions, text="Copy", command=self.copy_to_clipboard).pack(side="left", padx=5)
        ttk.Button(actions, text="Save Screenshot", command=self.save_screenshot).pack(side="left", padx=5)

        return tab

    def range_text(self, feature):
        stats = self.feature_stats[feature]
        return f"{fmt(stats['min'])} to {fmt(stats['max'])}"

    def check_ranges(self, event=None):
        for feature, label in self.range_labels.items():
            text = self.entries[feature].get().strip()
            stats = self.feature_stats[feature]
            try:
                value = float(text)
            except ValueError:
                label.config(text=f"{self.range_text(feature)}   invalid number", fg=self.colors["red"])
                continue
            if value < stats["min"] or value > stats["max"]:
                label.config(text=f"{self.range_text(feature)}   outside training range", fg=self.colors["red"])
            else:
                label.config(text=self.range_text(feature), fg=self.colors["muted"])

    def create_domain_tab(self):
        tab = ttk.Frame(self.notebook)

        top = tk.Frame(tab, bg=self.colors["background"])
        top.pack(fill="x", padx=15, pady=(15, 6))
        tk.Label(
            top,
            text="Validated Domain",
            font=("Segoe UI", 17, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["background"],
        ).pack(side="left")
        tk.Label(
            top,
            text="Predictions are supported only inside this domain",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["background"],
        ).pack(side="left", padx=15, pady=5)

        summary = self.card(tab)
        summary.pack(fill="x", padx=15, pady=6)
        lines = [
            f"Training mixtures: {len(self.X_train)}    Test mixtures: {len(self.X_test)}    "
            f"Inputs: {len(self.feature_names)}"
            + (f"    Excluded identifier columns: {', '.join(self.excluded_columns)}" if self.excluded_columns else ""),
            f"Univariate check: each numeric input must lie within its training minimum and maximum (table below).",
            f"Joint check: Euclidean distance to the nearest training mixture in standardised input space. "
            f"Caution above {fmt(self.nn_caution)} ({NN_CAUTION_PERCENTILE}th percentile of training "
            f"nearest-neighbour distances); extrapolation above {fmt(self.nn_limit)} ({NN_LIMIT_PERCENTILE}th percentile).",
            f"Prediction interval: {int((1 - ALPHA) * 100)}% cross-conformal from {CV_FOLDS}-fold out-of-fold "
            f"residuals, half-width {self.q:.3f} {UNIT}; empirical test coverage "
            f"{self.test_coverage * 100:.1f}% (n = {len(self.X_test)}).",
            f"Test mixtures beyond the extrapolation distance: {self.test_beyond_limit}; "
            f"outside a univariate range: {self.test_out_of_range}.",
        ]
        if self.age_feature:
            stats = self.feature_stats.get(self.age_feature, {})
            if stats.get("type") == "numeric":
                lines.append(
                    f"Curing age ({self.age_feature}): {fmt(stats['min'])} to {fmt(stats['max'])} days."
                )
        tk.Label(
            summary,
            text="\n".join(lines),
            font=("Segoe UI", 9),
            fg=self.colors["text"],
            bg=self.colors["card"],
            justify="left",
            anchor="w",
            wraplength=1300,
        ).pack(fill="x", padx=14, pady=12)

        table_card = self.card(tab)
        table_card.pack(fill="both", expand=True, padx=15, pady=(6, 12))

        columns = ("feature", "type", "range", "median", "mean")
        tree = ttk.Treeview(table_card, columns=columns, show="headings")
        headings = {
            "feature": ("Input", 320),
            "type": ("Type", 110),
            "range": ("Training range", 380),
            "median": ("Median / mode", 160),
            "mean": ("Mean", 160),
        }
        for key, (text, width) in headings.items():
            tree.heading(key, text=text)
            tree.column(key, width=width, anchor="w")

        for feature in self.feature_names:
            stats = self.feature_stats[feature]
            if stats["type"] == "numeric":
                tree.insert(
                    "",
                    "end",
                    values=(
                        feature,
                        "numeric",
                        f"{fmt(stats['min'])} to {fmt(stats['max'])}",
                        fmt(stats["median"]),
                        fmt(stats["mean"]),
                    ),
                )
            else:
                tree.insert(
                    "",
                    "end",
                    values=(feature, "categorical", ", ".join(stats["categories"]), stats["mode"], ""),
                )

        scrollbar = ttk.Scrollbar(table_card, orient="vertical", command=tree.yview)
        tree.configure(yscrollcommand=scrollbar.set)
        tree.pack(side="left", fill="both", expand=True, padx=10, pady=10)
        scrollbar.pack(side="right", fill="y", pady=10)

        return tab

    def create_analysis_tab(self):
        tab = ttk.Frame(self.notebook)

        top = tk.Frame(tab, bg=self.colors["background"])
        top.pack(fill="x", padx=15, pady=(15, 8))
        tk.Label(
            top,
            text="Model Analysis",
            font=("Segoe UI", 17, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["background"],
        ).pack(side="left")
        tk.Label(
            top,
            text="Held-out test set",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["background"],
        ).pack(side="left", padx=15, pady=5)

        metric_frame = tk.Frame(tab, bg=self.colors["background"])
        metric_frame.pack(fill="x", padx=15, pady=5)

        metrics = [
            ("R²", f"{self.test_r2:.4f}", "light_blue", "navy"),
            ("RMSE", f"{self.test_rmse:.3f}", "light_teal", "teal"),
            ("MAE", f"{self.test_mae:.3f}", "light_green", "green"),
            ("MAPE", f"{self.test_mape:.2f}%" if np.isfinite(self.test_mape) else "N/A", "light_orange", "orange"),
            ("PI coverage", f"{self.test_coverage * 100:.1f}%", "light_purple", "purple"),
            ("PI width", f"{self.interval_width:.2f}", "light_blue", "blue"),
        ]
        for title, value, bg, fg in metrics:
            self.metric_card(metric_frame, title, value, self.colors[bg], self.colors[fg]).pack(
                side="left", fill="x", expand=True, padx=5
            )

        controls = self.card(tab)
        controls.pack(fill="x", padx=15, pady=5)
        for text, command in [
            ("Actual vs Predicted", self.plot_actual_vs_predicted),
            ("Residuals", self.plot_residuals),
            ("Residual Histogram", self.plot_residual_histogram),
            ("Feature Importance", self.plot_feature_importance),
            ("Correlation Matrix", self.plot_correlation_matrix),
        ]:
            ttk.Button(controls, text=text, command=command, width=20).pack(side="left", padx=5, pady=10)

        self.analysis_frame = self.card(tab)
        self.analysis_frame.pack(fill="both", expand=True, padx=15, pady=10)
        self.plot_actual_vs_predicted()

        return tab

    def metric_card(self, parent, title, value, background, foreground):
        frame = tk.Frame(
            parent,
            bg=background,
            highlightbackground=self.colors["border"],
            highlightthickness=1,
        )
        tk.Label(frame, text=title, font=("Segoe UI", 9, "bold"), fg=self.colors["muted"], bg=background).pack(pady=(12, 0))
        tk.Label(frame, text=value, font=("Segoe UI", 17, "bold"), fg=foreground, bg=background).pack(pady=(2, 12))
        return frame

    def create_history_tab(self):
        tab = ttk.Frame(self.notebook)

        top = tk.Frame(tab, bg=self.colors["background"])
        top.pack(fill="x", padx=15, pady=15)
        tk.Label(
            top,
            text="Prediction History",
            font=("Segoe UI", 17, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["background"],
        ).pack(side="left")
        self.history_stats_var = tk.StringVar(value="No predictions yet")
        tk.Label(
            top,
            textvariable=self.history_stats_var,
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["background"],
        ).pack(side="right")

        controls = tk.Frame(tab, bg=self.colors["background"])
        controls.pack(fill="x", padx=15)
        ttk.Button(controls, text="Export CSV / Excel", command=self.export_history).pack(side="left", padx=4)
        ttk.Button(controls, text="Clear History", command=self.clear_history).pack(side="left", padx=4)
        tk.Label(
            controls,
            text="Double-click a row to reload it",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["background"],
        ).pack(side="left", padx=12)

        holder = self.card(tab)
        holder.pack(fill="both", expand=True, padx=15, pady=12)

        columns = ["time", "cs", "lower", "upper", "domain"]
        if self.age_feature:
            columns.insert(1, "age")
        self.history_tree = ttk.Treeview(holder, columns=columns, show="headings")
        labels = {
            "time": ("Time", 170),
            "age": ("Curing age (days)", 130),
            "cs": (f"CS ({UNIT})", 110),
            "lower": ("Lower 95%", 110),
            "upper": ("Upper 95%", 110),
            "domain": ("Domain check", 260),
        }
        for key in columns:
            text, width = labels[key]
            self.history_tree.heading(key, text=text)
            self.history_tree.column(key, width=width, anchor="w")

        scrollbar = ttk.Scrollbar(holder, orient="vertical", command=self.history_tree.yview)
        self.history_tree.configure(yscrollcommand=scrollbar.set)
        self.history_tree.pack(side="left", fill="both", expand=True, padx=10, pady=10)
        scrollbar.pack(side="right", fill="y", pady=10)
        self.history_tree.bind("<Double-Button-1>", self.load_history_entry)

        return tab

    def fill_sample(self, sample):
        for feature in self.feature_names:
            value = sample.get(feature, None)
            widget = self.entries[feature]
            if feature in self.categorical_features:
                if value is not None and pd.notna(value):
                    widget.set(str(value))
            else:
                widget.delete(0, tk.END)
                if value is None or pd.isna(value):
                    widget.insert(0, fmt(self.feature_stats[feature]["median"], 6))
                else:
                    widget.insert(0, fmt(value, 8))
        self.check_ranges()

    def fill_random_sample(self):
        index = np.random.randint(len(self.X_train))
        self.fill_sample(self.X_train.iloc[index].to_dict())

    def fill_min_case(self):
        self.fill_sample(self.X_train.loc[self.y_train.idxmin()].to_dict())

    def fill_max_case(self):
        self.fill_sample(self.X_train.loc[self.y_train.idxmax()].to_dict())

    def read_inputs(self):
        values = {}
        for feature in self.feature_names:
            text = self.entries[feature].get().strip()
            if feature in self.categorical_features:
                if text not in self.feature_stats[feature]["categories"]:
                    raise ValueError(f"Unknown category for '{feature}': {text}")
                values[feature] = text
            else:
                try:
                    number = float(text)
                except ValueError:
                    raise ValueError(f"Invalid numeric value for '{feature}'.")
                if not np.isfinite(number):
                    raise ValueError(f"Invalid value for '{feature}'.")
                values[feature] = number
        return values

    def assess_domain(self, frame):
        row = frame.iloc[0]
        flagged = self.out_of_range(row)
        distances, indices = self.nearest_distance(frame)
        distance = float(distances[0])
        nearest_cs = float(self.y_train.iloc[int(indices[0])])

        if flagged or distance > self.nn_limit:
            status = "Extrapolation"
        elif distance > self.nn_caution:
            status = "Sparse region"
        else:
            status = "Within validated domain"

        lines = []
        for feature, value, low, high in flagged:
            lines.append(
                f"{feature} = {fmt(value)} is outside the training range {fmt(low)} to {fmt(high)}."
            )
        if distance > self.nn_limit:
            lines.append(
                f"The input combination is farther from the training data than {NN_LIMIT_PERCENTILE}% "
                f"of training mixtures (distance {distance:.3f}, limit {self.nn_limit:.3f})."
            )
        elif distance > self.nn_caution:
            lines.append(
                f"The input combination lies in a sparsely sampled region (distance {distance:.3f}, "
                f"caution {self.nn_caution:.3f}, limit {self.nn_limit:.3f})."
            )
        else:
            lines.append(
                f"All inputs are inside the training ranges and close to training mixtures "
                f"(distance {distance:.3f}, caution {self.nn_caution:.3f})."
            )
        if status == "Extrapolation":
            lines.append("The prediction and its interval are not validated for this input.")
        elif status == "Sparse region":
            lines.append("Treat the prediction and its interval with caution.")
        lines.append(f"Nearest training mixture: CS = {nearest_cs:.2f} {UNIT}.")

        return {
            "status": status,
            "message": "\n".join(lines),
            "distance": distance,
            "flagged": [f[0] for f in flagged],
            "nearest_cs": nearest_cs,
        }

    def predict(self):
        try:
            values = self.read_inputs()
        except ValueError as exc:
            messagebox.showerror("Input error", str(exc))
            return

        frame = pd.DataFrame([values], columns=self.feature_names)
        prediction = float(self.pipeline.predict(frame)[0])
        lower = prediction - self.q
        upper = prediction + self.q
        domain = self.assess_domain(frame)

        entry = {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "CS": prediction,
            "lower_95": lower,
            "upper_95": upper,
            "domain_status": domain["status"],
            "nn_distance": domain["distance"],
            "out_of_range": ", ".join(domain["flagged"]),
            "domain_message": domain["message"],
            "inputs": values,
        }
        self.show_entry(entry)
        self.history.append(entry)
        self.save_history()
        self.update_history_display()
        self.status_var.set(f"Prediction completed | {domain['status']}")

        if domain["status"] == "Extrapolation":
            messagebox.showwarning("Outside validated domain", domain["message"])

    def show_entry(self, entry):
        self.current = entry
        self.result_var.set(f"{entry['CS']:.2f}")
        self.interval_var.set(
            f"{int((1 - ALPHA) * 100)}% prediction interval: "
            f"{entry['lower_95']:.2f} to {entry['upper_95']:.2f} {UNIT}"
        )
        if self.age_feature and self.age_feature in entry["inputs"]:
            self.age_var.set(f"Curing age: {fmt(entry['inputs'][self.age_feature])} days")
        else:
            self.age_var.set("")

        palette = {
            "Within validated domain": ("light_green", "green"),
            "Sparse region": ("light_orange", "orange"),
            "Extrapolation": ("light_red", "red"),
        }
        bg_key, fg_key = palette.get(entry["domain_status"], ("light_blue", "navy"))
        bg = self.colors[bg_key]
        self.domain_card.configure(bg=bg)
        self.domain_title.configure(text=entry["domain_status"], bg=bg, fg=self.colors[fg_key])
        self.domain_text.configure(text=entry.get("domain_message", ""), bg=bg)

    def clear_inputs(self):
        for feature in self.feature_names:
            stats = self.feature_stats[feature]
            widget = self.entries[feature]
            if stats["type"] == "categorical":
                widget.set(stats["mode"])
            else:
                widget.delete(0, tk.END)
                widget.insert(0, fmt(stats["median"], 6))
        self.check_ranges()
        self.current = None
        self.result_var.set("READY")
        self.interval_var.set("")
        self.age_var.set("")
        bg = self.colors["light_blue"]
        self.domain_card.configure(bg=bg)
        self.domain_title.configure(text="Domain check", bg=bg, fg=self.colors["navy"])
        self.domain_text.configure(
            text="Run a prediction to check the input against the validated domain.", bg=bg
        )
        self.status_var.set(f"Model ready ({self.model_source})")

    def update_history_display(self):
        self.history_tree.delete(*self.history_tree.get_children())
        for i, entry in enumerate(self.history):
            row = [entry["timestamp"]]
            if self.age_feature:
                row.append(fmt(entry["inputs"].get(self.age_feature)))
            row += [
                f"{entry['CS']:.2f}",
                f"{entry['lower_95']:.2f}",
                f"{entry['upper_95']:.2f}",
                entry.get("domain_status", ""),
            ]
            self.history_tree.insert("", "end", iid=str(i), values=row)

        if self.history:
            values = [e["CS"] for e in self.history]
            self.history_stats_var.set(
                f"{len(values)} predictions | Mean {np.mean(values):.2f} | "
                f"Min {np.min(values):.2f} | Max {np.max(values):.2f} {UNIT}"
            )
        else:
            self.history_stats_var.set("No predictions yet")

    def load_history_entry(self, event=None):
        selection = self.history_tree.selection()
        if not selection:
            return
        index = int(selection[0])
        if index >= len(self.history):
            return
        entry = self.history[index]
        self.fill_sample(entry["inputs"])
        self.show_entry(entry)
        self.notebook.select(0)

    def save_history(self):
        try:
            with open(self.history_path, "w", encoding="utf-8") as handle:
                json.dump(self.history, handle, indent=4)
        except Exception as exc:
            self.status_var.set(f"History could not be saved: {exc}")

    def load_history(self):
        if not os.path.exists(self.history_path):
            return
        try:
            with open(self.history_path, "r", encoding="utf-8") as handle:
                loaded = json.load(handle)
            self.history = [e for e in loaded if isinstance(e, dict) and "inputs" in e]
        except Exception:
            self.history = []
        self.update_history_display()

    def clear_history(self):
        if not self.history:
            return
        if messagebox.askyesno("Clear History", "Delete all prediction history?"):
            self.history = []
            self.save_history()
            self.update_history_display()

    def export_history(self):
        if not self.history:
            messagebox.showinfo("No Data", "No prediction history exists.")
            return
        filename = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx")],
        )
        if not filename:
            return
        rows = []
        for entry in self.history:
            row = {k: v for k, v in entry.items() if k not in ("inputs", "domain_message")}
            row.update(entry["inputs"])
            rows.append(row)
        table = pd.DataFrame(rows)
        if filename.lower().endswith(".xlsx"):
            table.to_excel(filename, index=False)
        else:
            table.to_csv(filename, index=False)
        messagebox.showinfo("Export Complete", "Prediction history exported.")

    def result_text(self):
        entry = self.current
        lines = [
            "COMPRESSIVE STRENGTH PREDICTION",
            "",
            f"Predicted CS: {entry['CS']:.2f} {UNIT}",
            f"{int((1 - ALPHA) * 100)}% prediction interval: "
            f"{entry['lower_95']:.2f} to {entry['upper_95']:.2f} {UNIT}",
        ]
        if self.age_feature:
            lines.append(f"Curing age: {fmt(entry['inputs'].get(self.age_feature))} days")
        lines += [
            f"Domain check: {entry['domain_status']}",
            entry.get("domain_message", ""),
            "",
            "INPUT PARAMETERS",
        ]
        for feature in self.feature_names:
            lines.append(f"{feature}: {entry['inputs'].get(feature)}")
        lines += [
            "",
            "MODEL",
            "Extra Trees regressor (scikit-learn pipeline)",
            ", ".join(f"{k}={v}" for k, v in ET_PARAMS.items() if k != "n_jobs"),
            f"Test R2 = {self.test_r2:.4f}, RMSE = {self.test_rmse:.3f}, MAE = {self.test_mae:.3f}",
            f"Interval half-width = {self.q:.3f} {UNIT}, test coverage = {self.test_coverage * 100:.1f}%",
            f"Data SHA-256: {self.data_hash}",
        ]
        return "\n".join(lines)

    def save_result(self):
        if self.current is None:
            messagebox.showinfo("No Prediction", "Generate a prediction first.")
            return
        filename = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt")],
            initialfile="CS_prediction.txt",
        )
        if not filename:
            return
        with open(filename, "w", encoding="utf-8") as handle:
            handle.write(self.result_text())
        messagebox.showinfo("Saved", "Prediction saved.")

    def copy_to_clipboard(self):
        if self.current is None:
            return
        self.root.clipboard_clear()
        self.root.clipboard_append(self.result_text())
        self.root.update()
        self.status_var.set("Result copied to clipboard")

    def save_screenshot(self):
        try:
            from PIL import ImageGrab
        except ImportError:
            messagebox.showerror("Screenshot", "Install Pillow to save screenshots (pip install pillow).")
            return
        filename = filedialog.asksaveasfilename(
            defaultextension=".png",
            filetypes=[("PNG image", "*.png")],
            initialfile="Fig14_GUI.png",
        )
        if not filename:
            return
        self.root.lift()
        self.root.update_idletasks()
        self.root.after(600, lambda: self.grab_window(ImageGrab, filename))

    def grab_window(self, grabber, filename):
        self.root.update()
        x, y = self.root.winfo_rootx(), self.root.winfo_rooty()
        w, h = self.root.winfo_width(), self.root.winfo_height()
        try:
            image = grabber.grab(bbox=(x, y, x + w, y + h), all_screens=True)
        except TypeError:
            image = grabber.grab(bbox=(x, y, x + w, y + h))
        image.save(filename, dpi=(300, 300))
        self.status_var.set(f"Screenshot saved: {filename}")

    def show_figure(self, fig):
        for widget in self.analysis_frame.winfo_children():
            widget.destroy()
        canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill="both", expand=True, padx=8, pady=8)

    def new_axes(self, size=(10, 6.5)):
        fig = Figure(figsize=size, dpi=100, facecolor=self.colors["card"])
        ax = fig.add_subplot(111)
        ax.set_facecolor(self.colors["card"])
        ax.grid(True, alpha=0.2)
        return fig, ax

    def plot_actual_vs_predicted(self):
        fig, ax = self.new_axes()
        y = self.y_test.to_numpy()
        p = self.y_test_pred
        low = min(y.min(), p.min())
        high = max(y.max(), p.max())
        line = np.array([low, high])
        ax.fill_between(line, line - self.q, line + self.q, color=self.colors["blue"], alpha=0.12,
                        label=f"{int((1 - ALPHA) * 100)}% interval band")
        ax.plot(line, line, linestyle="--", linewidth=1.6, color=self.colors["navy"], label="y = x")
        ax.scatter(y, p, s=45, alpha=0.8, color=self.colors["blue"], edgecolor="white", linewidth=0.5)
        ax.set_xlabel(f"Actual CS ({UNIT})", fontsize=11)
        ax.set_ylabel(f"Predicted CS ({UNIT})", fontsize=11)
        ax.legend(loc="lower right", frameon=False)
        ax.text(
            0.04, 0.96,
            f"R² = {self.test_r2:.4f}\nRMSE = {self.test_rmse:.3f}\nMAE = {self.test_mae:.3f}\n"
            f"Coverage = {self.test_coverage * 100:.1f}%\nn = {len(y)}",
            transform=ax.transAxes, va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.9),
        )
        fig.tight_layout()
        self.show_figure(fig)

    def plot_residuals(self):
        fig, ax = self.new_axes()
        residuals = self.y_test.to_numpy() - self.y_test_pred
        ax.scatter(self.y_test_pred, residuals, s=45, alpha=0.8, color=self.colors["teal"],
                   edgecolor="white", linewidth=0.5)
        ax.axhline(0, linestyle="--", linewidth=1.6, color=self.colors["navy"])
        ax.axhline(self.q, linestyle=":", linewidth=1.4, color=self.colors["red"])
        ax.axhline(-self.q, linestyle=":", linewidth=1.4, color=self.colors["red"])
        ax.set_xlabel(f"Predicted CS ({UNIT})", fontsize=11)
        ax.set_ylabel(f"Residual, actual minus predicted ({UNIT})", fontsize=11)
        fig.tight_layout()
        self.show_figure(fig)

    def plot_residual_histogram(self):
        fig, ax = self.new_axes()
        residuals = self.y_test.to_numpy() - self.y_test_pred
        ax.hist(residuals, bins=20, alpha=0.8, color=self.colors["blue"], edgecolor="white")
        ax.axvline(0, linestyle="--", linewidth=1.6, color=self.colors["navy"])
        ax.set_xlabel(f"Residual ({UNIT})", fontsize=11)
        ax.set_ylabel("Frequency", fontsize=11)
        fig.tight_layout()
        self.show_figure(fig)

    def plot_feature_importance(self):
        names = self.numeric_features + self.categorical_features
        importance = self.model.feature_importances_
        order = np.argsort(importance)
        fig, ax = self.new_axes(size=(10, max(4.5, 0.38 * len(names) + 1.5)))
        ax.barh(np.array(names)[order], importance[order], color=self.colors["teal"])
        ax.set_xlabel("Mean decrease in impurity", fontsize=11)
        ax.grid(True, axis="x", alpha=0.2)
        ax.grid(False, axis="y")
        fig.tight_layout()
        self.show_figure(fig)

    def plot_correlation_matrix(self):
        numeric = self.X_full[self.numeric_features].copy()
        numeric[TARGET] = self.y_full.values
        if numeric.shape[1] < 2:
            messagebox.showinfo("Correlation", "Not enough numeric variables.")
            return
        corr = numeric.corr()
        fig = Figure(figsize=(11, 9), dpi=100, facecolor=self.colors["card"])
        ax = fig.add_subplot(111)
        image = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
        labels = [str(c)[:18] for c in corr.columns]
        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels(labels, fontsize=8)
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        self.show_figure(fig)

    def on_closing(self):
        self.save_history()
        self.root.destroy()

    def run(self):
        self.root.mainloop()


if __name__ == "__main__":
    try:
        app = CompressiveStrengthApp()
    except Exception as exc:
        holder = tk.Tk()
        holder.withdraw()
        messagebox.showerror("Application error", str(exc))
        holder.destroy()
    else:
        app.run()